# MATH 70116 Deep Learning, Autumn 2026

# Single-layer, one-dimensional RNN

The input data will simply be a sequence of 1s and 0s, of length $T$, defined as a Markov process with 
$$P( x_{t+1} = 1 | x_t = 1) = p, \quad \quad P( x_{t+1} = 0 | x_t = 0) = q$$
Given $x_t$, we want to train the RNN to predict $x_{t+1}$.

Therefore we take the training data to be pairs $( x_t , y_t )$, with $y_t = x_{t+1}$.

In [24]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [3]:
def generate_markov_sequence(T, p, q, x0=None):
    """
    Generate a single Markov sequence of length T
    with parameters p, q
    optional initialisation of x[0]=x0
    """
    x = torch.zeros(T, dtype=torch.float32)
    x[0] = x0 if x0 is not None else torch.randint(0, 2, (1,)).float()
    for t in range(1, T):
        if x[t-1] == 1:
            x[t] = 1.0 if torch.rand(1) < p else 0.0
        else:
            x[t] = 0.0 if torch.rand(1) < q else 1.0
    return x

In [5]:
# sanity check
xx = generate_markov_sequence(100,0.5,0.5)
print(torch.sum(xx))

tensor(41.)


In [6]:
def make_batch(batch_size, T, p, q):
    """
    Returns:
        inputs  : (batch_size, T, 1)   : x_0, ..., x_{T-1}
        targets : (batch_size, T)      : x_1, ..., x_T  (integers for BCE)
    """
    seqs = torch.stack([generate_markov_sequence(T + 1, p, q)
                        for _ in range(batch_size)])   # (batch, T+1)
    inputs  = seqs[:, :-1].unsqueeze(-1)               # (batch, T, 1)
    targets = seqs[:, 1:]                              # (batch, T)
    return inputs, targets

In [7]:
class SingleLayerRNN(nn.Module):
    """
    Single-layer RNN with hidden dimension 1.
    At each step:
        a_t = tanh(u * a_{t-1} + w * x_t + b)
        y_t_hat = sigmoid(alpha * a_t + beta)
    """
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=1,
            num_layers=1,
            batch_first=True,   # input shape: (batch, seq, feature)
        )
        self.output = nn.Linear(1, 1)

    def forward(self, x, h0=None):
        # x   : (batch, T, 1)
        # h0  : (1, batch, 1)  or None (defaults to zeros)
        h_seq, h_last = self.rnn(x, h0)   # h_seq: (batch, T, 1)
        logits = self.output(h_seq)        # (batch, T, 1)
        return logits.squeeze(-1)          # (batch, T)

In [8]:
# training loop

def train(p=0.8, q=0.7, T=20, batch_size=128, n_steps=2000, lr=0.01):

    model = SingleLayerRNN()
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    for step in range(n_steps):
        inputs, targets = make_batch(batch_size, T, p, q)

        logits = model(inputs)                          # (batch, T)
        loss = loss_fn(logits, targets)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        if (step + 1) % 10 == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                # Separate predictions by current input value
                mask_1 = (inputs.squeeze(-1) == 1)   # positions where x_t = 1
                mask_0 = (inputs.squeeze(-1) == 0)   # positions where x_t = 0
                pred_given_1 = probs[mask_1].mean().item()
                pred_given_0 = probs[mask_0].mean().item()

            print(f"Step {step+1:4d} | Loss {loss.item():.4f} | "
                  f"P(y=1|x=1) = {pred_given_1:.3f} (target {p:.3f}) | "
                  f"P(y=1|x=0) = {pred_given_0:.3f} (target {1-q:.3f})")

    return model

In [9]:
p, q = 0.5, 0.1
print(f"Markov chain: p={p}, q={q}")
print(f"Optimal predictions: P(y=1|x=1)={p}, P(y=1|x=0)={1-q}\n")
model = train(p=p, q=q)

# Inspect learned parameters
print("\nLearned parameters:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.data.squeeze().item():.4f}")

Markov chain: p=0.5, q=0.1
Optimal predictions: P(y=1|x=1)=0.5, P(y=1|x=0)=0.9

Step   10 | Loss 0.9999 | P(y=1|x=1) = 0.218 (target 0.500) | P(y=1|x=0) = 0.265 (target 0.900)
Step   20 | Loss 0.8684 | P(y=1|x=1) = 0.293 (target 0.500) | P(y=1|x=0) = 0.341 (target 0.900)
Step   30 | Loss 0.7769 | P(y=1|x=1) = 0.369 (target 0.500) | P(y=1|x=0) = 0.403 (target 0.900)
Step   40 | Loss 0.7260 | P(y=1|x=1) = 0.434 (target 0.500) | P(y=1|x=0) = 0.455 (target 0.900)
Step   50 | Loss 0.6922 | P(y=1|x=1) = 0.489 (target 0.500) | P(y=1|x=0) = 0.501 (target 0.900)
Step   60 | Loss 0.6713 | P(y=1|x=1) = 0.533 (target 0.500) | P(y=1|x=0) = 0.541 (target 0.900)
Step   70 | Loss 0.6597 | P(y=1|x=1) = 0.567 (target 0.500) | P(y=1|x=0) = 0.574 (target 0.900)
Step   80 | Loss 0.6557 | P(y=1|x=1) = 0.592 (target 0.500) | P(y=1|x=0) = 0.598 (target 0.900)
Step   90 | Loss 0.6522 | P(y=1|x=1) = 0.608 (target 0.500) | P(y=1|x=0) = 0.616 (target 0.900)
Step  100 | Loss 0.6460 | P(y=1|x=1) = 0.618 (target 0.5

In [10]:
# Inspect learned parameters
print("\nLearned parameters:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.data.squeeze().item():.4f}")


Learned parameters:
  rnn.weight_ih_l0: -1.3632
  rnn.weight_hh_l0: -0.0012
  rnn.bias_ih_l0: 0.9151
  rnn.bias_hh_l0: 0.3185
  output.weight: 2.2606
  output.bias: 0.2927


In [11]:
print(model)

SingleLayerRNN(
  (rnn): RNN(1, 1, batch_first=True)
  (output): Linear(in_features=1, out_features=1, bias=True)
)


### Exercise: evaluate on some new data

In [20]:
# test on some new data
p=0.8
q=0.7
T = 1000

x_new = generate_markov_sequence(T, p, q)
print(type(x_new), x_new.shape)

<class 'torch.Tensor'> torch.Size([1000])
